# 第 30 天：终极项目

> 所属阶段：把单个因子研究升级为完整量化研究项目
> 今日主题：终极项目
> 必做：完整因子研究报告
> 选做：制作PPT
> 目标产出：面试级项目


> “ 项目段公共环境”：第21-30天共享统一的研究数据和工具箱。
> 请先阅读并运行**第21天**的「准备统一研究环境」（第3节）和「准备统一研究工具箱」（第4节）。
> 本日文件仅展示当日独有的实验内容，公共代码不再重复。


## 0. 今天你要真正学会什么？

1. 形成一份完整因子研究报告结构。
2. 生成项目指标表、图表清单、PPT 大纲和面试讲稿。
3. 把研究从代码结果升级为可展示、可复盘、可追问的项目。

今天不是孤立知识点，而是终局项目的一块拼图。  
你要把前面学过的标签、IC、分层、中性化、标准化、Alpha101 和回测方法接起来。


## 1. 先建立直觉

终极项目不是把前 29 天的东西简单拼起来，而是讲清楚一个完整研究故事：我为什么做、怎么做、发现了什么、风险在哪里、下一步怎么改。

## 5. 今日核心实验


### 实验 1：项目总览：从研究问题开始，不从代码开始

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
project_brief = pd.Series({
    "研究问题": "多因子模型能否在模拟股票池中提供稳定的横截面选股能力？",
    "股票池": f"{len(assets)} 只模拟股票",
    "样本区间": f"{dates.min().date()} 至 {dates.max().date()}",
    "标签": "未来 5 日收益",
    "核心方法": "因子预处理 + IC 检验 + 多因子合成 + 组合回测 + 风险归因",
})

print(project_brief)


### 实验 2：候选因子体检表

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
candidate_names = ["value", "quality", "momentum_20", "low_vol", "liquidity", "reversal_5", "price_volume"]
candidate_library = {name: factor_library[name] for name in candidate_names}
candidate_report = summarize_library(candidate_library, future_5d)
print(candidate_report[["ic_mean", "ic_ir", "ic_positive_ratio", "turnover"]].round(4))


### 实验 3：形成最终模型并回测

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
selected = candidate_report.query("turnover < 0.60").index[:5].tolist()
final_score = make_equal_weight_composite(candidate_library, selected)
final_weights = make_market_neutral_weights(final_score, q=0.2)
final_ret = portfolio_return(final_weights, returns)

final_stats = pd.Series({
    "selected_factors": ", ".join(selected),
    "mean_return": final_ret.mean(),
    "vol": final_ret.std(),
    "sharpe_like": final_ret.mean() / final_ret.std() * np.sqrt(252) if final_ret.std() else np.nan,
    "win_rate": (final_ret > 0).mean(),
    "max_drawdown": ((1 + final_ret).cumprod() / (1 + final_ret).cumprod().cummax() - 1).min(),
    "avg_turnover": final_weights.diff().abs().sum(axis=1).mean(),
})

print(final_stats)


### 实验 4：生成报告图表

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
(1 + final_ret.fillna(0)).cumprod().plot(ax=axes[0], title="终极项目：策略净值")
rank_ic(final_score, future_5d).dropna().cumsum().plot(ax=axes[1], title="终极项目：累计 Rank IC")
for ax in axes:
    ax.axhline(0 if ax is axes[1] else 1, color="black", linewidth=1)
plt.tight_layout()
plt.show()
plt.close()


### 实验 5：PPT 大纲和面试讲稿

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
ppt_outline = pd.DataFrame([
    [1, "研究问题", "为什么做多因子模型，目标是什么"],
    [2, "数据与标签", "股票池、样本期、未来收益标签、避免未来函数"],
    [3, "因子库", "价值、质量、动量、波动率、流动性、Alpha101"],
    [4, "因子检验", "IC、ICIR、分层收益、换手"],
    [5, "模型构建", "预处理、等权合成、权重逻辑"],
    [6, "回测结果", "净值、收益风险、回撤、胜率"],
    [7, "风险归因", "风格暴露、行业暴露、残差"],
    [8, "失败和限制", "成本、样本外、拥挤、参数敏感性"],
    [9, "下一步", "真实数据、滚动训练、交易成本和组合优化"],
], columns=["页码", "标题", "要讲清楚的事"])

print(ppt_outline)

script = """
我的项目不是寻找一个神奇公式，而是搭建一条完整的因子研究流水线。
我先用未来收益构造标签，再对候选因子做去极值、标准化和 IC 检验。
之后我选择稳定性较好的因子做等权合成，并用多空组合检验排序能力。
最后我检查换手、回撤和风险暴露，避免把风格或行业暴露误认为 Alpha。
"""
print(script.strip())


## 6. 结果应该怎么写进研究笔记？

建议你用下面这个模板记录今天的内容：


主题：终极项目
研究问题：
使用数据：
核心方法：
关键指标：
最重要的图：
结果是否稳定：
主要风险：
是否进入下一步：


写研究笔记时，尽量少写“效果不错”这种空话。  
你要写得像另一个研究员下周接手还能继续做：

- 指标是多少。
- 参数是什么。
- 样本区间是什么。
- 你为什么选择这个方法。
- 你发现了什么限制。

## 7. 常见坑深挖

### 坑 1：报告只贴图，不解释研究问题。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 2：只展示好结果，不展示失败和风险控制。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 3：PPT 像流水账，没有主线故事。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 4：面试时只会说工具，不会说判断和取舍。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

## 8. 今日验收标准

完成今天课程后，你应该能拿出这些产出：

- 一张核心结果表。
- 一张能解释研究结论的图。
- 一段自然语言结论。
- 一条明确的下一步动作：保留、观察、改造或淘汰。
- 至少一个你亲手检查过的风险点。

## 9. 强化练习

### 作业 A：复述今日方法

不用公式，用自然语言讲清楚今天的方法解决了什么问题。

### 作业 B：换一个参数

改变一个窗口、阈值或因子集合，观察结论是否改变。

### 作业 C：写风险说明

至少写出三个风险：

1. 数据风险
2. 模型风险
3. 交易风险

### 作业 D：连接前面的课程

写出今天内容和第 1-20 天中哪三天关系最密切。

## 10. 面试式自测

### 问 1：今天的方法解决的核心问题是什么？

答案：它把单个研究动作放进更完整的因子研究流水线，帮助判断信号是否可用、稳定、可解释。

### 问 2：为什么不能只看收益曲线？

答案：收益曲线可能来自样本内过拟合、行业暴露、风格暴露或偶然市场环境，必须同时检查 IC、换手、稳定性和风险来源。

### 问 3：今天的结果如果不好，是否代表方法无效？

答案：不一定。坏结果也能提供信息，关键是判断问题来自因子本身、参数选择、数据质量，还是市场状态。

### 问 4：今天最容易出现的未来函数在哪里？

答案：通常出现在权重估计、模型训练、标签构造和调参选择中。只要用了未来才知道的信息，结果就不可信。

## 11. 今日复盘模板


我今天最理解的概念：

我今天最容易混淆的地方：

我跑出的关键表格：

我跑出的关键图：

我对结果的判断：

我发现的风险：

我明天要继续的问题：


## 12. 下一课连接

30 天计划结束后，你可以继续进入真实数据、实盘约束和研究自动化。

## 13. 一句话收尾

终极项目 的重点不是技巧本身，而是把技巧放进可复现、可解释、可迭代的研究系统里。

## 14. 学习提醒（更新版）

30天计划结束不是终点，而是真实研究的起点。下一步：用第0天准备的真实数据替换所有模拟数据，重新跑一遍完整流程。

---

## 15. 终极项目：面试级因子研究报告模板

> 以下是一份可以直接交给面试官或导师的完整报告结构。替换 `[...]` 后即可使用。

### 15.1 报告结构


# 因子研究报告：[你的研究标题]

## 1. 研究摘要（200字）
- 研究问题
- 核心发现
- 关键数字（IC均值、ICIR、多空年化收益、Sharpe）

## 2. 数据说明
- 数据来源（AKShare / Tushare / Wind）
- 样本区间（2019-01 至 2024-12）
- 股票池（沪深300 / 中证500 / 全A）
- 数据预处理（复权方式、停牌处理、ST过滤）

## 3. 因子构建
- 因子定义（公式 + 经济直觉）
- 预处理流程（去极值 → 行业中性化 → 市值中性化 → Z-score标准化）
- 因子分布特征

## 4. 单因子检验
- Rank IC 统计（均值、IR、正比例）
- IC 序列图
- 5分组分层回测收益曲线
- 多空组合净值曲线
- 分年度表现

## 5. 多因子合成与ML模型
- 因子相关性矩阵
- 合成方法（等权 / IC加权 / Ridge / LGB）
- 样本外IC对比
- 特征重要性分析

## 6. 策略回测
- 组合构建方法
- 扣成本后净值曲线
- 风险指标（年化收益、波动率、Sharpe、最大回撤、Calmar）
- 分年度收益表
- 行业暴露分析
- 换手率分析

## 7. 风险与限制
- 过拟合风险（训练/测试IC差距）
- 幸存者偏差
- 容量限制
- 市场状态依赖

## 8. 结论与改进方向
- 核心结论（3句话）
- 下一步改进（3个方向）
- 是否建议实盘


### 15.2 图表清单（必做）

| 编号 | 图表 | 内容 |
|------|------|------|
| 图1 | IC序列图 | 时间序列 + 滚动12月均值线 |
| 图2 | 分层收益曲线 | 5组累计收益 + 多空组合 |
| 图3 | 因子相关性热力图 | 所有候选因子的Spearman相关矩阵 |
| 图4 | 多空净值曲线 | 扣成本前后对比 |
| 图5 | 回撤图 | 滚动252日最大回撤 |
| 图6 | 分年度收益柱状图 | 每年收益 + 基准对比 |

### 15.3 面试讲稿模板（5分钟版）


【开场 30秒】
我完成了一个A股多因子选股研究项目。
样本是2019-2024年沪深300成分股，构建了价值、质量、动量、波动率4大类共[8]个因子。

【方法论 1分钟】
每个因子经过统一的预处理流程：去极值、行业中性化、市值中性化、Z-score标准化。
用Rank IC和5分组回测检验因子有效性。

【核心发现 1分钟】
等权合成因子Rank IC均值[0.035]，ICIR[0.45]。
多空组合年化收益[12%]，Sharpe[1.2]。
其中动量因子IC最强([0.04])，但波动率因子稳定性最好(ICIR[0.6])。

【ML增强 1分钟】
用Ridge回归和LightGBM做多因子合成，样本外IC从[0.035]提升到[0.042]。
通过Purged TimeSeries CV和过拟合诊断确认提升是真实的。

【风险与改进 1分钟】
主要风险：样本区间较短（5年）、未考虑交易成本后的实际收益下滑约[2%]年化。
改进方向：加入另类数据因子、引入风险模型做组合优化、在更长样本期验证。


### 15.4 关键指标速查表

| 指标 | 目标值 | 你的值 |
|------|--------|--------|
| Rank IC均值 | > 0.02 | [填入] |
| ICIR | > 0.3 | [填入] |
| 多空年化收益 | > 8% | [填入] |
| Sharpe比率 | > 1.0 | [填入] |
| 最大回撤 | < 25% | [填入] |
| 单边日换手率 | < 30% | [填入] |
| 训练/测试IC差距 | < 0.03 | [填入] |

---

## 16. 进阶：从项目到实盘的差距清单

> 以下每一项都是从"研究项目"到"实盘策略"必须跨越的鸿沟。

- [ ] **真实数据替换**：用第0天准备的真实行情+财务数据完全替换模拟数据
- [ ] **股票池动态管理**：处理成分股调入调出，而非固定股票池
- [ ] **停牌/涨跌停处理**：停牌日不能交易，涨跌停日无法以收盘价成交
- [ ] **交易成本实算**：用逐笔成本替代固定费率
- [ ] **冲击成本建模**：大单对市场价格的推动效应
- [ ] **容量测算**：策略最大可管理资金规模
- [ ] **信号衰减分析**：因子从计算到执行之间的延迟效应
- [ ] **绩效归因**：Brinson归因或因子归因，拆解收益来源
- [ ] **风控系统**：止损、仓位上限、行业偏离度约束
- [ ] **复盘机制**：定期检查策略是否偏离预期行为

---

## 17. 推荐下一步学习资源

### 必读论文
- Fama & French (1993): Common risk factors in the returns on stocks and bonds
- Fama & French (2015): A five-factor asset pricing model
- Kakushadze (2016): 101 Formulaic Alphas
- Gu, Kelly & Xiu (2020): Empirical Asset Pricing via Machine Learning

### 推荐书籍
- 《因子投资：方法与实践》（石川 等）
- 《Active Portfolio Management》（Grinold & Kahn）
- 《Advances in Financial Machine Learning》（Marcos López de Prado）

### 推荐工具
- **Alphalens**（Quantopian）：因子分析一站式工具
- **PyPortfolioOpt**：组合优化库
- **Zipline** / **Backtrader**：回测框架
- **riskfolio-lib**：风险模型和组合优化

---

本课程内容仅用于量化研究学习，不构成投资建议。
